### Optimizer(최적화 알고리즘)
: 손실함수를 통계 계산한 기울기를 이용하여 신경망의 weight와 bias를 업데이트하는 알고리즘
- 신경망 학습의 순서
    1. 입력 데이터를 모델에 전달
    2. 모델이 예측값을 계산
    3. 손실함수로 예측값과 정답의 차이를 계산
    4. loss.backward()로 각 파라미터의 gradient를 계산
    5. optimizer.step()으로 파라미터 업데이트
- 주요 구성 요소

|구성요소|역할|
|:---:|:---:|
|Loss function|예측값과 정답의 차이 계산|
|Gradient|파라미터를 조금 변경했을 때 손실이 변하는 방향과 정도|
|Optimizer|Gradient를 이용하여 파라미터 업데이트|
|Learning Rate|한번 업데이트할 때 변경하는 크기 조절|
#### Gradient Dencent(경사하강법)
: 손실함수가 작아지는 방향으로 파라미터를 조금씩 변경하는 방법
- gradient가 양수면 파라미터를 감소, 음수면 증가시키는 방향으로 업데이트
- 손실이 증가하는 방향을 나타내므로 gredient의 반대 방향으로 파라미터를 이동시켜 손실을 최소화
- 용어설명
    - 파라미터:(parameter): 모델이 학습을 통해 조정하는 모든 값
    - 가중치(weight): 입력 데이터의 영향력을 조절하는 파라미터
    - 편향(bias): 입력과 별개로 출력값을 조정하는 파라미터

In [1]:
import torch
import torch.nn as nn
import torch.optim as op

In [2]:
# 간단한 파라미터와 손실 함수
w = torch.tensor(2.0, requires_grad=True)
tg = torch.tensor(10.0)
pd = w * 3
loss = (pd - tg) ** 2
loss.backward()
print("예측값:", pd.item())
print("손실값:", loss.item())
print("weight의 gradient:", w.grad.item())
print("현재의 weight:", w.item())

예측값: 6.0
손실값: 16.0
weight의 gradient: -24.0
현재의 weight: 2.0


In [3]:
w_old = w.item()
gd = w.grad.item()
lr = 0.1
w_new = w_old - lr * gd
print("업데이터 전:", w_old)
print("gradient:", gd)
print("learning rate:", lr)
print("업데이트 후:", w_new)

업데이터 전: 2.0
gradient: -24.0
learning rate: 0.1
업데이트 후: 4.4


In [4]:
w = torch.tensor(2.0, requires_grad=True)
tg = torch.tensor(10.0)
lr = 0.1
for i in range(2):
    # 예측 및 손실 계산
    pd = w * 3
    loss = (pd - tg) ** 2
    # gradient 계산
    loss.backward()
    print(pd.item(), loss.item(), w.grad.item(), w.item())
    # 파라미터 업데이트
    with torch.no_grad():
        w -= lr * w.grad
    print(w)
    # 다음학습을 위해 gradient 초기화
    w.grad.zero_()

6.0 16.0 -24.0 2.0
tensor(4.4000, requires_grad=True)
13.200000762939453 10.240004539489746 19.20000457763672 4.400000095367432
tensor(2.4800, requires_grad=True)


#### Learning rate(학습률)
: 파라미터를 한번 업데이트 할 때 얼마나 크게 변경할 것인지를 결정하는 값
**Learning rate가 너무 작은 경우**
    - 파라미터가 조금씩 변경됨
    - 손실이 감소하는 속도가 느릴 수 있음
    - 학습에 많은 반복이 필요할 수 있음
**Learning rate가 너무 큰 경우**
    - 파라미터가 지나치게 크게 변경됨
    - 최적값을 지나쳐 손실이 오히려 커질 수 있음
    - 학습이 불안정하거나 발산할 수 있음
- 하이퍼 파라미터(사용자가 넣음): 모델과 데이터에 맞게 조정해야 하는 중요한 하이퍼 파라미터

|Optimizer|일반적인 시작 학습률(Learning rate)|
|:---:|:---:|
|경사하강법, SGD|0.01|
|SGD+Momentum|0.01~0.1|
|Adam|0.001|
|AdamW|0.001|
|RMSporp|0.001|

In [5]:
# Gradient가 동일할 때 Learning rate에 따른 업데이트 크기 비교
w=5.0
gd=2.0
for i in [0.001, 0.1,0.5]:
    uw=w-i*gd
    print('업뎃후:',w)

업뎃후: 5.0
업뎃후: 5.0
업뎃후: 5.0


#### SGD(stochastic gradient descent)
: gradient를 이용하여 파라미터를 업데이트 하는 기본적인 최적화 알고리즘
- 사용형식
    - torch.optim.SGD(model.parameter(), lr=0.01)
- 특징
    - 구조가 단순하고 이해하기 쉬움
    - learning rate 설정에 영향을 많이 받음
    - momentum 옵션을 사용하여 이전 업데이트 방향을 반영할 수 있음
- 기본값은 momentum을 사용하지 않음, 사용하려면 별도 지정 필요

In [7]:
model = nn.Linear(1, 1)
optimizer = op.SGD(model.parameters(), lr=0.01)
print(optimizer, model.weight, model.bias)

SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
) Parameter containing:
tensor([[-0.2785]], requires_grad=True) Parameter containing:
tensor([0.6584], requires_grad=True)


#### optimizer.step()
: backward()로 계산한 gradient를 이용해 모델의 파라미터를 업데이트하는 메서드
- 일반적인 순서
    - optimizer.zero_grad(): 이전 gradient 제거
    - prediction=model(x): 예측값 계산
    - loss=loss_fn(prediction, target): 손실 계산
    - loss.backward(): gradient 계산
    - optimizer.step(): 파라미터 업데이트

**중요!**<br>
optimizer.step() 호출 전에 먼저 loss.backward()를 먼저 실행해야 현재 손실의 gradient를 이용할 수 있음